In [28]:
from qiskit.quantum_info import DensityMatrix, Kraus, partial_trace, Pauli

In [3]:
from numpy import sqrt
1/sqrt(2)

0.7071067811865475

In [2]:
def damp_err(gamma, n):
    '''This method produces noise operators
    gamma [float]: Damping probability
    n [int]: Number of qubit
    '''
    
    if not isinstance(n, int) or n <= 0:
        raise ValueError(f"Number of qubit should be positive integer. Given {n}")
    if not isinstance(gamma, float) or gamma < 0 or gamma > 1:
        raise ValueError(f"Damping probability should lie between 0 and 1. Given {gamma}")
        
    from numpy import eye, zeros, kron, sqrt
    
    _E = [eye(2), zeros((2, 2))]
    _E[0][1][1] = sqrt(1-gamma)
    _E[1][0][1] = sqrt(gamma)
    
    E = list(kron(_E, eye(2**(~-n)))/sqrt(n))
    for m in range(1, n):
        E += list(kron(eye(2**m), kron(_E, eye(2**(~-n-m))))/sqrt(n))
    return E

In [32]:
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))))
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))).expand(DensityMatrix([1, 0])))
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))).expand(DensityMatrix([1, 0])).measure([0, 1])[0], partial_trace(DensityMatrix([0, 0, 1, 0]).evolve(Kraus(damp_err(0.0, 2))), [0, 1]))
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))).expand(DensityMatrix([1, 0])).measure([1])[0], partial_trace(DensityMatrix([0, 0, 1, 0]).evolve(Kraus(damp_err(0.0, 2))), [1]))
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))).expand(DensityMatrix([1, 0])).measure([0])[0], partial_trace(DensityMatrix([0, 0, 1, 0]).evolve(Kraus(damp_err(0.0, 2))), [0]))
# DensityMatrix(DensityMatrix([0, 1]).data)

DensityMatrix([[0.+0.j, 0.+0.j],
               [0.+0.j, 1.+0.j]],
              dims=(2,))
DensityMatrix([[0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
               [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j],
               [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
               [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]],
              dims=(2, 2))
01 DensityMatrix([[1.+0.j]],
              dims=())
0 DensityMatrix([[1.+0.j, 0.+0.j],
               [0.+0.j, 0.+0.j]],
              dims=(2,))
1 DensityMatrix([[0.+0.j, 0.+0.j],
               [0.+0.j, 1.+0.j]],
              dims=(2,))


In [26]:
def enc(n, k, d):
    if [n, k, d] == [5, 1, 3]:
        s0 = [0, 18, 9, 20, 10, -27, -6, -24, -29, -3, -30, -15, -17, -12, -23, 5]
        s1 = [31, 13, 22, 11, 21, -4, -25, -7, -2, -28, -1, -16, -14, -19, -8, 26]
        enc_op = zeros((2, 2**n))
        for i in s0:
            v = 0.25
            if i != abs(i):
                v = -v
            enc_op[0][abs(i)] = v
        for i in s1:
            v = 0.25
            if i != abs(i):
                v = -v
            enc_op[1][abs(i)] = v
        return Operator(transpose(enc_op))

In [27]:
from qiskit.quantum_info import Operator
from numpy import transpose, matmul, zeros

mat = enc(5, 1, 3).data
# mat
matmul(transpose(mat), mat)

array([[1.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j]])